# 01 - Pipeline ETL para el TFG
Este notebook realiza la extracción, transformación y carga de los datos de detecciones acústicas y trazas GPS.


## 1. Carga y Consolidación de Predicciones
Leemos todos los archivos `predicciones_*.txt` de la carpeta `raw` y los unificamos en un solo DataFrame.


In [1]:
import pandas as pd
from pathlib import Path

# Asegurar directorios
Path("../data/processed").mkdir(parents=True, exist_ok=True)
Path("../outputs").mkdir(parents=True, exist_ok=True)

COLS = ["microfono_id", "t_start", "t_end", "class", "confidence"]

# Búsqueda flexible: predicciones.txt, Predicciones_..., predicciones_...
dfs = []
for f in Path("../data/raw").rglob("*"):
    if f.is_file() and "predicciones" in f.name.lower() and f.suffix == ".txt":
        day = f.parent.name
        # on_bad_lines='skip' ignora filas que tengan un número incorrecto de columnas (visto en logs: 9 en vez de 5)
        df = pd.read_csv(f, header=None, names=COLS, on_bad_lines='skip')
        df["date"] = day
        dfs.append(df)

if dfs:
    pred = pd.concat(dfs, ignore_index=True)
    
    # --- FUNCIONALIDAD DE CORRECCIÓN HORARIA ---
    # Si algún día el dispositivo tenía la hora mal (p.ej. no se actualizó el cambio de hora),
    # indicamos aquí las horas que hay que SUMAR o RESTAR a ese día concreto.
    # Ejemplo: {"11-03-2026": -1} restará una hora a todas las predicciones de ese día.
    TIME_CORRECTIONS = {
        #"11-03-2026": -1,
        "23-03-2026": -1,
    }
    
    for day_corr, hours in TIME_CORRECTIONS.items():
        mask = pred["date"] == day_corr
        if mask.any():
            print(f"Aplicando corrección de {hours}h al día {day_corr}")
            pred.loc[mask, "t_start"] = pd.to_datetime(pred.loc[mask, "t_start"], format="mixed") + pd.Timedelta(hours=hours)
            pred.loc[mask, "t_end"] = pd.to_datetime(pred.loc[mask, "t_end"], format="mixed") + pd.Timedelta(hours=hours)
    # -------------------------------------------
    
    # Validación y Limpieza Profunda
    # 1. Eliminar espacios y tabulaciones en columnas de texto
    for col in pred.select_dtypes(include=['object']).columns:
        pred[col] = pred[col].astype(str).str.strip().replace(r'\\t', '', regex=True)
        
    # 2. Eliminar filas vacías o con NaNs en columnas clave
    pred = pred.dropna(subset=COLS)
    
    # 3. Eliminar duplicados
    duplicados = pred.duplicated().sum()
    if duplicados > 0:
        print(f"Eliminando {duplicados} filas duplicadas.")
        pred = pred.drop_duplicates()
    
    # Localizar como Europe/Madrid (local) y convertir a UTC para comparar con GPS
    pred["t_start"] = pd.to_datetime(pred["t_start"], format="mixed")
    pred["t_end"]   = pd.to_datetime(pred["t_end"], format="mixed")
    
    pred["t_start"] = pred["t_start"].dt.tz_localize("Europe/Madrid", ambiguous='infer').dt.tz_convert("UTC")
    pred["t_end"]   = pred["t_end"].dt.tz_localize("Europe/Madrid", ambiguous='infer').dt.tz_convert("UTC")
    
    pred["duration_s"] = (pred["t_end"] - pred["t_start"]).dt.total_seconds()
    pred["class"] = pred["class"].astype(int)
    pred.to_parquet("../data/processed/predictions.parquet", index=False)
    print(f"Predicciones procesadas: {len(pred)} filas.")
else:
    print("No se encontraron archivos de predicción.")

Aplicando corrección de -1h al día 23-03-2026
Predicciones procesadas: 14323 filas.


## 2. Carga de Trazas GPS
Procesamos los archivos `.gpx` para extraer coordenadas (lat, lon, ele) y marcas de tiempo.


In [2]:
import gpxpy

rows = []
for f in Path("../data/raw").rglob("*.gpx"):
    day = f.parent.name
    direction = "ida" if "_15_" in f.stem or "_16_" in f.stem else "vuelta"
    with open(f) as fh:
        gpx = gpxpy.parse(fh)
    for track in gpx.tracks:
        for seg in track.segments:
            for pt in seg.points:
                rows.append({
                    "date": day,
                    "direction": direction,
                    "lat": pt.latitude,
                    "lon": pt.longitude,
                    "ele": pt.elevation,
                    "time": pt.time,
                })

if rows:
    tracks = pd.DataFrame(rows)
    tracks["time"] = pd.to_datetime(tracks["time"], utc=True)
    tracks.to_parquet("../data/processed/tracks.parquet", index=False)
    print(f"Puntos GPS procesados: {len(tracks)} filas.")
else:
    print("No se encontraron archivos GPX.")


Puntos GPS procesados: 17488 filas.


## 3.5. Validación de Sincronía Temporal
Antes de realizar el cruce, comprobamos que los rangos horarios de las predicciones y los GPS coinciden. Esto es vital para asegurar que la localización sea válida.


In [3]:
if 'tracks' in locals() and 'pred' in locals():
    print("Validación de solapamiento temporal (UTC) por día:")
    
    # Rangos GPS
    gps_res = tracks.groupby("date")["time"].agg(["min", "max"]).rename(columns={"min": "gps_start", "max": "gps_end"})
    
    # Rangos Predicciones
    pred_res = pred.groupby("date")["t_start"].agg(["min", "max"]).rename(columns={"min": "pred_start", "max": "pred_end"})
    
    # Unir
    check = pd.concat([gps_res, pred_res], axis=1)
    
    # Calcular solapamiento en minutos
    overlap_start = pd.concat([check["gps_start"], check["pred_start"]], axis=1).max(axis=1)
    overlap_end = pd.concat([check["gps_end"], check["pred_end"]], axis=1).min(axis=1)
    check["overlap_min"] = (overlap_end - overlap_start).dt.total_seconds() / 60
    
    # Formatear para visualización legíble
    summary = check.copy()
    for col in ["gps_start", "gps_end", "pred_start", "pred_end"]:
        summary[col] = summary[col].dt.strftime('%H:%M')
    
    display(summary)
    
    # Alertas
    dias_criticos = check[check["overlap_min"] <= 0].index.tolist()
    if dias_criticos:
        print(f"\n⚠️ ALERTA: Sin solapamiento temporal en: {dias_criticos}")
        print("Revisar si la conversión Europe/Madrid -> UTC es correcta o si los archivos son del día correcto.")
    else:
        print("\n✅ Éxito: Todas las predicciones tienen solapamiento temporal con los GPS.")
else:
    print("No hay suficientes datos para validar.")


Validación de solapamiento temporal (UTC) por día:


,gps_start,gps_end,pred_start,pred_end,overlap_min
date,,,,,
01-04-2026,12:48,18:24,12:46,18:24,335.682301
11-03-2026,13:59,19:38,20:16,20:37,-38.121661
14-04-2026,15:09,18:07,15:07,18:07,177.866667
15-04-2026,12:38,17:30,12:36,17:29,290.618651
16-04-2026,12:52,17:58,12:49,17:58,306.050000
20-04-2026,12:23,17:19,12:21,17:19,296.204429
22-04-2026,13:10,18:33,13:08,18:32,322.326985
23-03-2026,14:10,18:42,14:08,18:42,271.200000
23-04-2026,12:45,18:47,12:41,18:47,361.828451



⚠️ ALERTA: Sin solapamiento temporal en: ['11-03-2026']
Revisar si la conversión Europe/Madrid -> UTC es correcta o si los archivos son del día correcto.


## 4. Join Espacio-Temporal
Asignamos a cada evento acústico la coordenada GPS más cercana en el tiempo para su geolocalización.


In [4]:
import numpy as np

if 'tracks' in locals() and 'pred' in locals():
    tracks_sorted = tracks.sort_values("time")
    pred_sorted   = pred.sort_values("t_start")

    def nearest_gps(row, gps_day):
        if gps_day.empty:
            return np.nan, np.nan
        t_mid = row["t_start"] + (row["t_end"] - row["t_start"]) / 2
        diffs = (gps_day["time"] - t_mid).abs()
        if diffs.min().total_seconds() > 4:
            return np.nan, np.nan
        idx = diffs.idxmin()
        return gps_day.loc[idx, "lat"], gps_day.loc[idx, "lon"]

    lats, lons = [], []
    for _, row in pred_sorted.iterrows():
        day_str = row["date"]
        gps_day = tracks_sorted[tracks_sorted["date"] == day_str]
        lat, lon = nearest_gps(row, gps_day)
        lats.append(lat); lons.append(lon)

    pred_sorted["lat"] = lats
    pred_sorted["lon"] = lons
    pred_geo = pred_sorted.dropna(subset=["lat", "lon"])
    pred_geo.to_parquet("../data/processed/predictions_geo.parquet", index=False)
    print(f"Predicciones geolocalizadas con éxito: {len(pred_geo)}.")


Predicciones geolocalizadas con éxito: 9293.


## 5. Validación Final del ETL
Comprobamos la calidad del cruce de datos y nos aseguramos de que solo se han unido predicciones dentro de las ventanas temporales de los GPS.


In [5]:
if 'pred_geo' in locals() and 'pred' in locals():
    print("Resumen de Calidad del Join (por día):")
    
    # Métricas por día
    daily_total = pred.groupby("date").size().rename("Total Preds")
    daily_geo = pred_geo.groupby("date").size().rename("Joined Preds")
    
    # Ventanas temporales
    all_pred_window = pred.groupby("date")["t_start"].agg(lambda x: f"{x.min().strftime('%H:%M')} - {x.max().strftime('%H:%M')}").rename("Ventana Preds (UTC)")
    gps_window = tracks.groupby("date")["time"].agg(lambda x: f"{x.min().strftime('%H:%M')} - {x.max().strftime('%H:%M')}").rename("Ventana GPS (UTC)")
    pred_window = pred_geo.groupby("date")["t_start"].agg(lambda x: f"{x.min().strftime('%H:%M')} - {x.max().strftime('%H:%M')}").rename("Ventana Join (UTC)")
    
    # Consolidar tabla
    resumen_final = pd.concat([daily_total, daily_geo, all_pred_window, gps_window, pred_window], axis=1).fillna(0)
    resumen_final["% Geolocalizado"] = (resumen_final["Joined Preds"] / resumen_final["Total Preds"] * 100).round(1).astype(str) + "%"
    
    display(resumen_final)
    
    print(f"\nGeolocalización exitosa: {len(pred_geo)} de {len(pred)} eventos ({len(pred_geo)/len(pred):.1%}).")
    print("Las predicciones descartadas suelen ser pruebas realizadas antes/después de los trayectos GPS.")
    
    # Desglose por micrófono
    print("\nDetecciones geolocalizadas por micrófono:")
    print(pred_geo["microfono_id"].value_counts())
else:
    print("Error: No se han generado los datos geolocalizados.")


Resumen de Calidad del Join (por día):


,Total Preds,Joined Preds,Ventana Preds (UTC),Ventana GPS (UTC),Ventana Join (UTC),% Geolocalizado
date,,,,,,
01-04-2026,1628,998.0,12:46 - 18:24,12:48 - 18:24,12:48 - 18:24,61.3%
11-03-2026,442,0.0,20:16 - 20:37,13:59 - 19:38,0,0.0%
14-04-2026,1450,928.0,15:07 - 18:07,15:09 - 18:07,15:09 - 18:07,64.0%
15-04-2026,1396,832.0,12:36 - 17:29,12:38 - 17:30,12:39 - 17:29,59.6%
16-04-2026,1324,829.0,12:49 - 17:58,12:52 - 17:58,12:52 - 17:58,62.6%
20-04-2026,2464,1597.0,12:21 - 17:19,12:23 - 17:19,12:22 - 17:19,64.8%
22-04-2026,1021,705.0,13:08 - 18:32,13:10 - 18:33,13:10 - 18:32,69.0%
23-03-2026,934,677.0,14:08 - 18:42,14:10 - 18:42,14:10 - 18:42,72.5%
23-04-2026,1121,833.0,12:41 - 18:47,12:45 - 18:47,12:45 - 18:47,74.3%



Geolocalización exitosa: 9293 de 14323 eventos (64.9%).
Las predicciones descartadas suelen ser pruebas realizadas antes/después de los trayectos GPS.

Detecciones geolocalizadas por micrófono:
microfono_id
2.0    4774
1.0    4519
Name: count, dtype: int64
